In [ ]:
import os
import json
import re
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch

# ------------------------------------------------------------
# Verify the Python environment and GPU availability.
# ------------------------------------------------------------

print("Environment check")
print("=" * 50)

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU: CPU only")

print("=" * 50)
print("Environment setup completed successfully.")

In [ ]:
# ============================================================
# Step 2 - Download the Resume Dataset
# ============================================================
#
# The dataset contains annotated resumes that we will use as
# our candidate database for the talent search engine.
#
# We are using the KaggleHub API instead of manually uploading
# the dataset so the experiment is reproducible.
# ============================================================

import kagglehub
import os

# Download the official Resume Entities for NER dataset.
dataset_path = kagglehub.dataset_download(
    "dataturks/resume-entities-for-ner"
)

print("Dataset downloaded successfully.")
print("Dataset path:", dataset_path)

# Display the files downloaded from Kaggle.
print("\nDataset files:")
for root, dirs, files in os.walk(dataset_path):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
# ============================================================
# Step 3 - Load and Inspect the Dataset
# ============================================================
#
# The dataset is stored in JSON Lines (JSONL) format.
# Each line represents one resume containing:
#   - content: the full resume text
#   - annotation: labeled entities extracted from the resume
#   - extras: additional metadata when available
#
# We first load the records and inspect one example before
# building the RAG pipeline.
# ============================================================

json_path = os.path.join(
    dataset_path,
    "Entity Recognition in Resumes.json"
)

data = []

# Read the file line by line because it is stored as JSONL.
with open(json_path, "r", encoding="utf-8") as f:
    for line_number, line in enumerate(f, start=1):
        line = line.strip()

        # Skip empty lines.
        if not line:
            continue

        # Convert each JSON line into a Python dictionary.
        data.append(json.loads(line))

print("Dataset loaded successfully.")
print("Number of resume records:", len(data))

# ------------------------------------------------------------
# Inspect the first record.
# ------------------------------------------------------------

print("\nFirst record structure:")
print("Keys:", list(data[0].keys()))

print("\nFirst resume text preview:")
print(data[0]["content"][:1000])

print("\nNumber of annotations in first resume:",
      len(data[0].get("annotation", [])))

In [ ]:
# ============================================================
# Step 4 - Analyze Resume Entity Types
# ============================================================
#
# The dataset contains manually annotated entities such as:
# skills, job titles, companies, education, location, etc.
#
# Before preprocessing the resumes, we need to understand
# which entity types are actually available in the dataset.
# This helps us design reliable candidate metadata and later
# validate semantic search results.
# ============================================================

entity_counter = Counter()

# Count every annotated entity type across all resumes.
for record in data:
    for annotation in record.get("annotation", []):

        # The label field may contain one or more entity types.
        labels = annotation.get("label", [])

        for label in labels:
            entity_counter[label] += 1

# ------------------------------------------------------------
# Display the dataset-level entity statistics.
# ------------------------------------------------------------

print("Number of resumes:", len(data))
print("Number of unique entity types:", len(entity_counter))

print("\nEntity distribution:")
print("-" * 40)

for entity, count in entity_counter.most_common():
    print(f"{entity}: {count}")

In [ ]:
# ============================================================
# Step 5 - Analyze Resume Lengths
# ============================================================
#
# Resume length directly affects our retrieval strategy.
#
# If resumes are long, representing each resume with a single
# embedding can hide important details such as specific skills,
# projects, or work experience.
#
# We therefore inspect the character distribution before
# deciding how to split resumes into retrieval chunks.
# ============================================================

resume_lengths = []

# Measure the character length of every resume.
for record in data:
    content = record.get("content", "")
    resume_lengths.append(len(content))

# ------------------------------------------------------------
# Dataset-level statistics
# ------------------------------------------------------------

print("Resume length statistics")
print("-" * 40)

print("Minimum characters:", min(resume_lengths))
print("Maximum characters:", max(resume_lengths))
print("Average characters:", round(np.mean(resume_lengths), 2))
print("Median characters:", round(np.median(resume_lengths), 2))
print("Total characters:", sum(resume_lengths))

# ------------------------------------------------------------
# Additional percentiles
# ------------------------------------------------------------
# Percentiles help us understand the typical resume size
# without being dominated by unusually long resumes.

print("\nPercentiles:")
print("25th percentile:", round(np.percentile(resume_lengths, 25), 2))
print("75th percentile:", round(np.percentile(resume_lengths, 75), 2))
print("90th percentile:", round(np.percentile(resume_lengths, 90), 2))
print("95th percentile:", round(np.percentile(resume_lengths, 95), 2))

In [ ]:
# ============================================================
# Step 6 - Build Structured Candidate Records
# ============================================================
#
# Each raw dataset record contains the resume text and its
# annotated entities.
#
# We convert the raw records into a cleaner candidate-level
# structure that will be used throughout the RAG pipeline.
#
# Each candidate keeps:
#   - candidate_id: stable internal identifier
#   - text: original resume content
#   - entities: extracted information from annotations
#
# We keep the original resume text at this stage so that no
# information is lost before retrieval and evaluation.
# ============================================================

resume_records = []

# Process every resume and assign a stable candidate ID.
for candidate_id, record in enumerate(data, start=1):

    # Extract and clean the original resume text.
    content = record.get("content", "").strip()

    # Store annotated entities grouped by their label.
    entity_data = defaultdict(list)

    for annotation in record.get("annotation", []):

        labels = annotation.get("label", [])
        points = annotation.get("points", [])

        for label in labels:
            for point in points:

                # Extract the actual entity text.
                text = point.get("text", "").strip()

                if text:
                    entity_data[label].append(text)

    # Create the structured candidate record.
    resume_records.append({
        "candidate_id": candidate_id,
        "text": content,
        "entities": dict(entity_data)
    })

# ------------------------------------------------------------
# Validate the resulting candidate collection.
# ------------------------------------------------------------

print("Candidate records created:", len(resume_records))

print("\nFirst candidate")
print("-" * 40)
print("Candidate ID:", resume_records[0]["candidate_id"])
print("Resume characters:", len(resume_records[0]["text"]))
print("Entity types:", list(resume_records[0]["entities"].keys()))

In [ ]:
# ============================================================
# Step 7 - Split Resumes into Retrieval Chunks
# ============================================================
#
# Long resumes are divided into smaller overlapping chunks.
#
# Why?
# A recruiter query may match a specific part of a resume,
# such as a technical skill, job title, or work experience.
# Chunk-level embeddings allow the vector database to retrieve
# that relevant section instead of representing the entire
# resume with a single vector.
#
# Configuration:
#   chunk_size    = 1000 characters
#   chunk_overlap = 150 characters
#
# The overlap preserves some context between neighboring chunks.
# ============================================================

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150

chunks = []


def split_text_into_chunks(text, chunk_size=1000, overlap=150):
    """
    Split a resume into overlapping text chunks.

    The function first tries to split at natural boundaries
    such as paragraphs, lines, and spaces. If a suitable
    boundary is not available, it falls back to a hard split.
    """

    # Return the complete text if it is already small enough.
    if len(text) <= chunk_size:
        return [text]

    chunks = []

    start = 0

    while start < len(text):

        # Define the initial end position.
        end = min(start + chunk_size, len(text))

        # If this is not the final chunk, try to find a natural
        # boundary close to the target chunk size.
        if end < len(text):

            # Look for the last paragraph/line boundary.
            boundary_positions = [
                text.rfind("\n\n", start, end),
                text.rfind("\n", start, end),
                text.rfind(". ", start, end),
                text.rfind(" ", start, end)
            ]

            # Select the closest valid natural boundary.
            valid_boundaries = [
                position for position in boundary_positions
                if position > start
            ]

            if valid_boundaries:
                end = max(valid_boundaries)

        # Extract and clean the chunk.
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        # Stop if we reached the end of the resume.
        if end >= len(text):
            break

        # Move the starting position backward to preserve context.
        next_start = end - overlap

        # Safety check to prevent an infinite loop.
        if next_start <= start:
            next_start = end

        start = next_start

    return chunks


# ------------------------------------------------------------
# Generate chunks for every candidate.
# ------------------------------------------------------------

for resume in resume_records:

    resume_chunks = split_text_into_chunks(
        resume["text"],
        chunk_size=CHUNK_SIZE,
        overlap=CHUNK_OVERLAP
    )

    # Attach candidate-level metadata to every chunk.
    for chunk_id, chunk_text in enumerate(resume_chunks, start=1):

        chunks.append({
            "candidate_id": resume["candidate_id"],
            "chunk_id": chunk_id,
            "text": chunk_text
        })


# ------------------------------------------------------------
# Validate the chunking result.
# ------------------------------------------------------------

print("Chunking completed successfully.")
print("-" * 50)

print("Number of candidates:", len(resume_records))
print("Total chunks:", len(chunks))
print(
    "Average chunks per candidate:",
    round(len(chunks) / len(resume_records), 2)
)

print("\nFirst chunk")
print("-" * 50)
print("Candidate ID:", chunks[0]["candidate_id"])
print("Chunk ID:", chunks[0]["chunk_id"])
print("Characters:", len(chunks[0]["text"]))

print("\nChunk text:")
print(chunks[0]["text"])

In [ ]:
# ============================================================
# Step 8 - Remove Direct Contact Information
# ============================================================
#
# The raw resumes contain direct contact information such as
# email addresses and Indeed profile URLs.
#
# These details are not useful for semantic talent matching.
# We therefore remove common contact patterns from the text
# that will be embedded into the vector database.
#
# Important:
# The original dataset and original resume text remain unchanged.
# We only create a cleaned version for retrieval and embeddings.
# ============================================================

def clean_resume_text(text):
    """
    Remove common direct contact information from resume text.
    """

    # Remove email addresses.
    text = re.sub(
        r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
        " ",
        text
    )

    # Remove common Indeed profile URLs.
    text = re.sub(
        r"https?://(?:www\.)?indeed\.com/[^\s]+",
        " ",
        text,
        flags=re.IGNORECASE
    )

    # Remove standalone Indeed profile references.
    text = re.sub(
        r"\bindeed\.com/[^\s]+",
        " ",
        text,
        flags=re.IGNORECASE
    )

    # Normalize repeated whitespace created by the removals.
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


# ------------------------------------------------------------
# Apply PII cleaning to every retrieval chunk.
# ------------------------------------------------------------

for chunk in chunks:
    chunk["retrieval_text"] = clean_resume_text(chunk["text"])


# ------------------------------------------------------------
# Validate the cleaning process on the first chunk.
# ------------------------------------------------------------

print("PII cleaning completed.")
print("-" * 50)

print("Original characters:",
      len(chunks[0]["text"]))

print("Cleaned characters:",
      len(chunks[0]["retrieval_text"]))

print("\nCleaned first chunk:")
print(chunks[0]["retrieval_text"])

In [ ]:
# ============================================================
# Step 9 - Finalize Contact Information Cleaning
# ============================================================
#
# The previous cleaning step removed the actual Indeed URL,
# but the surrounding label "Email me on Indeed:" remained.
#
# This step removes that leftover contact label as well.
# The goal is to keep retrieval focused on professional content
# such as skills, experience, education, and job roles.
# ============================================================

def clean_resume_text(text):
    """
    Remove direct contact information and related labels
    from resume text while preserving professional content.
    """

    # Remove email addresses.
    text = re.sub(
        r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
        " ",
        text
    )

    # Remove HTTP/HTTPS Indeed URLs.
    text = re.sub(
        r"https?://(?:www\.)?indeed\.com/[^\s]+",
        " ",
        text,
        flags=re.IGNORECASE
    )

    # Remove standalone Indeed profile URLs.
    text = re.sub(
        r"\b(?:www\.)?indeed\.com/[^\s]+",
        " ",
        text,
        flags=re.IGNORECASE
    )

    # Remove common contact labels left after URL removal.
    text = re.sub(
        r"\bEmail me on Indeed\s*:?",
        " ",
        text,
        flags=re.IGNORECASE
    )

    # Normalize whitespace.
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


# Rebuild the cleaned retrieval text for every chunk.
for chunk in chunks:
    chunk["retrieval_text"] = clean_resume_text(chunk["text"])


# ------------------------------------------------------------
# Validate the first cleaned chunk.
# ------------------------------------------------------------

print("Final contact cleaning completed.")
print("-" * 50)

print("Original characters:",
      len(chunks[0]["text"]))

print("Cleaned characters:",
      len(chunks[0]["retrieval_text"]))

print("\nCleaned first chunk:")
print(chunks[0]["retrieval_text"])

In [ ]:
import re

# Define patterns for detecting direct contact information
email_pattern = re.compile(
    r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"
)

indeed_pattern = re.compile(
    r"\b(?:https?://)?(?:www\.)?indeed\.com/[^\s]+",
    flags=re.IGNORECASE
)

email_matches = []
indeed_matches = []

for chunk in chunks:
    retrieval_text = chunk["retrieval_text"]

    email_found = email_pattern.findall(retrieval_text)
    indeed_found = indeed_pattern.findall(retrieval_text)

    if email_found:
        email_matches.extend(
            [(chunk["candidate_id"], chunk["chunk_id"], value) for value in email_found]
        )

    if indeed_found:
        indeed_matches.extend(
            [(chunk["candidate_id"], chunk["chunk_id"], value) for value in indeed_found]
        )

print("=" * 60)
print("PII VALIDATION")
print("=" * 60)

print(f"Total chunks checked: {len(chunks)}")
print(f"Email addresses found: {len(email_matches)}")
print(f"Indeed URLs found: {len(indeed_matches)}")

if not email_matches and not indeed_matches:
    print("\nValidation PASSED")
    print("No email addresses or Indeed URLs remain in retrieval_text.")
else:
    print("\nValidation FAILED")

    if email_matches:
        print("\nEmail matches:")
        for item in email_matches[:10]:
            print(item)

    if indeed_matches:
        print("\nIndeed URL matches:")
        for item in indeed_matches[:10]:
            print(item)

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

# Select GPU when CUDA is available, otherwise fall back to CPU.
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Embedding device: {device}")

if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Load a lightweight English sentence embedding model.
# The model is suitable for semantic search over English resumes.
embedding_model_name = "BAAI/bge-small-en-v1.5"

embedding_model = SentenceTransformer(
    embedding_model_name,
    device=device
)

print("\nEmbedding model loaded successfully.")
print(f"Model: {embedding_model_name}")
print(f"Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}")

In [ ]:
import numpy as np
from tqdm.auto import tqdm

# Extract cleaned resume chunks for embedding generation.
texts = [
    chunk["retrieval_text"]
    for chunk in chunks
]

print("=" * 60)
print("GENERATING RESUME CHUNK EMBEDDINGS")
print("=" * 60)

print(f"Texts to embed: {len(texts)}")
print(f"Embedding model: {embedding_model_name}")
print(f"Expected embedding dimension: {embedding_model.get_embedding_dimension()}")

# Generate normalized embeddings.
# Normalization allows FAISS Inner Product to represent cosine similarity.
embeddings = embedding_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

# Convert to float32 for FAISS compatibility and memory efficiency.
embeddings = embeddings.astype("float32")

print("\nEmbedding generation completed.")
print(f"Embedding matrix shape: {embeddings.shape}")
print(f"Embedding dtype: {embeddings.dtype}")

# Verify that the embeddings are normalized.
embedding_norms = np.linalg.norm(embeddings, axis=1)

print(f"Minimum embedding norm: {embedding_norms.min():.6f}")
print(f"Maximum embedding norm: {embedding_norms.max():.6f}")

In [ ]:
# Install FAISS for vector similarity search.
# The CPU package is sufficient because FAISS IndexFlatIP
# performs efficiently for our current 1,034-vector index.

!pip install -q faiss-cpu

print("faiss-cpu installation completed.")

In [ ]:
import faiss

# Read the embedding dimension from the generated embeddings.
embedding_dimension = embeddings.shape[1]

# Create an Inner Product index.
# Because embeddings are normalized, Inner Product is equivalent to cosine similarity.
faiss_index = faiss.IndexFlatIP(embedding_dimension)

# Add all resume chunk embeddings to the FAISS index.
faiss_index.add(embeddings)

print("=" * 60)
print("FAISS VECTOR DATABASE")
print("=" * 60)

print(f"Index type: IndexFlatIP")
print(f"Embedding dimension: {embedding_dimension}")
print(f"Vectors added: {faiss_index.ntotal}")
print(f"Expected vectors: {len(chunks)}")

# Verify that the FAISS index contains the expected number of vectors.
if faiss_index.ntotal == len(chunks):
    print("\nFAISS index validation PASSED.")
else:
    print("\nFAISS index validation FAILED.")

In [ ]:
def semantic_search(query, top_k=10):
    """
    Retrieve the most semantically relevant resume chunks for a recruiter query.

    Args:
        query: Natural-language recruiter query.
        top_k: Number of chunks to retrieve from FAISS.

    Returns:
        List of retrieved chunks with similarity scores and metadata.
    """

    # Generate a normalized embedding for the recruiter query.
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype("float32")

    # Search the FAISS index using cosine similarity.
    scores, indices = faiss_index.search(query_embedding, top_k)

    results = []

    for score, index in zip(scores[0], indices[0]):
        # Ignore invalid FAISS indices.
        if index == -1:
            continue

        result = chunks[index].copy()
        result["similarity_score"] = float(score)

        results.append(result)

    return results


# Run an initial recruiter-style semantic search.
test_query = "Junior Data Analyst with SQL and Tableau experience"

search_results = semantic_search(
    test_query,
    top_k=10
)

print("=" * 60)
print("SEMANTIC SEARCH TEST")
print("=" * 60)

print(f"Query: {test_query}")
print(f"Results returned: {len(search_results)}")

for rank, result in enumerate(search_results, start=1):
    print("\n" + "-" * 60)
    print(f"Rank: {rank}")
    print(f"Candidate ID: {result['candidate_id']}")
    print(f"Chunk ID: {result['chunk_id']}")
    print(f"Similarity: {result['similarity_score']:.4f}")
    print(f"Text preview: {result['retrieval_text'][:500]}")

In [ ]:
from collections import defaultdict


def retrieve_candidates(query, top_k_chunks=30, top_k_candidates=5):
    """
    Retrieve relevant candidates from the resume vector database.

    The search is performed at chunk level first, then results are grouped
    by candidate. Each candidate is represented by the highest similarity
    score among their retrieved chunks.

    Args:
        query: Natural-language recruiter query.
        top_k_chunks: Number of resume chunks retrieved from FAISS.
        top_k_candidates: Number of unique candidates to return.

    Returns:
        Ranked list of candidates with their relevant chunks and scores.
    """

    # Retrieve semantically relevant resume chunks.
    chunk_results = semantic_search(
        query=query,
        top_k=top_k_chunks
    )

    # Group retrieved chunks by candidate ID.
    candidate_groups = defaultdict(list)

    for result in chunk_results:
        candidate_id = result["candidate_id"]
        candidate_groups[candidate_id].append(result)

    candidate_results = []

    for candidate_id, candidate_chunks in candidate_groups.items():

        # Use the strongest matching chunk as the candidate's retrieval score.
        best_score = max(
            chunk["similarity_score"]
            for chunk in candidate_chunks
        )

        # Sort the candidate's retrieved chunks by relevance.
        candidate_chunks = sorted(
            candidate_chunks,
            key=lambda x: x["similarity_score"],
            reverse=True
        )

        candidate_results.append({
            "candidate_id": candidate_id,
            "similarity_score": best_score,
            "matched_chunks": candidate_chunks
        })

    # Rank unique candidates by their strongest semantic match.
    candidate_results.sort(
        key=lambda x: x["similarity_score"],
        reverse=True
    )

    return candidate_results[:top_k_candidates]


# Test candidate-level retrieval with the same recruiter query.
candidate_results = retrieve_candidates(
    query=test_query,
    top_k_chunks=30,
    top_k_candidates=5
)

print("=" * 60)
print("CANDIDATE-LEVEL RETRIEVAL")
print("=" * 60)

print(f"Query: {test_query}")
print(f"Unique candidates returned: {len(candidate_results)}")

for rank, candidate in enumerate(candidate_results, start=1):
    print("\n" + "-" * 60)
    print(f"Rank: {rank}")
    print(f"Candidate ID: {candidate['candidate_id']}")
    print(f"Best similarity: {candidate['similarity_score']:.4f}")
    print(f"Matched chunks: {len(candidate['matched_chunks'])}")

    for chunk in candidate["matched_chunks"][:2]:
        print(
            f"  Chunk {chunk['chunk_id']} "
            f"(score={chunk['similarity_score']:.4f})"
        )

In [ ]:
# Build a lookup table for the original candidate records.
# The original resume text is preserved separately from the cleaned retrieval text.
candidate_profiles = {
    resume["candidate_id"]: resume
    for resume in resume_records
}


def get_candidate_profile(candidate_id):
    """
    Return the original resume profile for a candidate.

    Args:
        candidate_id: Unique candidate identifier.

    Returns:
        Candidate profile containing the original resume text and extracted entities.
    """

    profile = candidate_profiles.get(candidate_id)

    if profile is None:
        raise ValueError(
            f"Candidate ID {candidate_id} was not found."
        )

    return profile


# Attach the full original resume profile to each retrieved candidate.
for candidate in candidate_results:
    profile = get_candidate_profile(candidate["candidate_id"])

    candidate["resume_text"] = profile["text"]
    candidate["entities"] = profile["entities"]


print("=" * 60)
print("CANDIDATE PROFILE RETRIEVAL")
print("=" * 60)

print(f"Candidate profiles available: {len(candidate_profiles)}")

for rank, candidate in enumerate(candidate_results, start=1):
    print("\n" + "-" * 60)
    print(f"Rank: {rank}")
    print(f"Candidate ID: {candidate['candidate_id']}")
    print(f"Similarity: {candidate['similarity_score']:.4f}")
    print(f"Resume characters: {len(candidate['resume_text'])}")
    print(f"Entity types: {list(candidate['entities'].keys())}")

In [ ]:
def build_llm_context(candidate_results, top_n=3):
    """
    Build grounded context for LLM evaluation.

    The context contains only information available in the resume.
    Direct contact information is removed before sending the content
    to the LLM.
    """

    llm_context = []

    for rank, candidate in enumerate(candidate_results[:top_n], start=1):

        # Clean the original resume before sending it to the LLM.
        cleaned_resume = clean_resume_text(
            candidate["resume_text"]
        )

        # Keep only non-contact entity information.
        safe_entities = {
            key: values
            for key, values in candidate["entities"].items()
            if key not in {"Email Address"}
        }

        # Keep the most relevant retrieved chunks as explicit evidence.
        evidence_chunks = [
            {
                "chunk_id": chunk["chunk_id"],
                "similarity_score": chunk["similarity_score"],
                "text": chunk["retrieval_text"]
            }
            for chunk in candidate["matched_chunks"][:3]
        ]

        llm_context.append({
            "rank": rank,
            "candidate_id": candidate["candidate_id"],
            "retrieval_score": candidate["similarity_score"],
            "resume_text": cleaned_resume,
            "entities": safe_entities,
            "evidence_chunks": evidence_chunks
        })

    return llm_context


# Build the context for the three highest-ranked candidates.
top_3_context = build_llm_context(
    candidate_results,
    top_n=3
)

print("=" * 60)
print("TOP 3 LLM CONTEXT")
print("=" * 60)

print(f"Candidates prepared: {len(top_3_context)}")

for candidate in top_3_context:
    print("\n" + "-" * 60)
    print(f"Rank: {candidate['rank']}")
    print(f"Candidate ID: {candidate['candidate_id']}")
    print(f"Retrieval score: {candidate['retrieval_score']:.4f}")
    print(f"Resume characters: {len(candidate['resume_text'])}")
    print(f"Evidence chunks: {len(candidate['evidence_chunks'])}")
    print(f"Entity types: {list(candidate['entities'].keys())}")

    print("\nResume preview:")
    print(candidate["resume_text"][:300].replace("\n", " "))

In [ ]:
import re
from difflib import SequenceMatcher


def normalize_for_comparison(text):
    """
    Normalize resume text for duplicate-profile comparison.

    The normalization removes direct contact information and
    non-essential formatting differences while preserving the
    resume content.
    """

    # Remove direct contact information before comparison.
    text = clean_resume_text(text)

    # Normalize whitespace and case for comparison.
    text = re.sub(r"\s+", " ", text).strip().lower()

    return text


def compare_candidate_profiles(candidate_a, candidate_b):
    """
    Compare two candidate resumes using normalized text similarity.
    """

    text_a = normalize_for_comparison(candidate_a["resume_text"])
    text_b = normalize_for_comparison(candidate_b["resume_text"])

    similarity = SequenceMatcher(
        None,
        text_a,
        text_b
    ).ratio()

    return similarity


print("=" * 60)
print("DUPLICATE PROFILE VALIDATION")
print("=" * 60)

for i in range(len(top_3_context)):
    for j in range(i + 1, len(top_3_context)):

        candidate_a = top_3_context[i]
        candidate_b = top_3_context[j]

        similarity = compare_candidate_profiles(
            candidate_a,
            candidate_b
        )

        print(
            f"Candidate {candidate_a['candidate_id']} "
            f"vs Candidate {candidate_b['candidate_id']}: "
            f"{similarity:.4f}"
        )

In [ ]:
DUPLICATE_THRESHOLD = 0.95


def deduplicate_candidates(candidate_results, similarity_threshold=0.95):
    """
    Remove near-duplicate candidate profiles from retrieval results.

    Candidates are compared in retrieval rank order.
    When two profiles exceed the duplicate similarity threshold,
    the later profile is excluded from the retrieval results.

    The original dataset remains unchanged.
    """

    unique_candidates = []

    for candidate in candidate_results:

        is_duplicate = False

        for existing_candidate in unique_candidates:

            similarity = compare_candidate_profiles(
                candidate,
                existing_candidate
            )

            if similarity >= similarity_threshold:
                is_duplicate = True
                break

        if not is_duplicate:
            unique_candidates.append(candidate)

    return unique_candidates


# Deduplicate the candidate-level retrieval results.
deduplicated_candidates = deduplicate_candidates(
    candidate_results,
    similarity_threshold=DUPLICATE_THRESHOLD
)

print("=" * 60)
print("DEDUPLICATED CANDIDATE RETRIEVAL")
print("=" * 60)

print(f"Candidates before deduplication: {len(candidate_results)}")
print(f"Candidates after deduplication: {len(deduplicated_candidates)}")
print(f"Duplicate threshold: {DUPLICATE_THRESHOLD}")

for rank, candidate in enumerate(deduplicated_candidates, start=1):
    print("\n" + "-" * 60)
    print(f"Rank: {rank}")
    print(f"Candidate ID: {candidate['candidate_id']}")
    print(f"Similarity: {candidate['similarity_score']:.4f}")

In [ ]:
# Build the final grounded context from deduplicated candidates.
final_top_3_context = build_llm_context(
    deduplicated_candidates,
    top_n=3
)

print("=" * 60)
print("FINAL TOP 3 LLM CONTEXT")
print("=" * 60)

print(f"Candidates prepared: {len(final_top_3_context)}")

for candidate in final_top_3_context:
    print("\n" + "-" * 60)
    print(f"Rank: {candidate['rank']}")
    print(f"Candidate ID: {candidate['candidate_id']}")
    print(f"Retrieval score: {candidate['retrieval_score']:.4f}")
    print(f"Resume characters: {len(candidate['resume_text'])}")
    print(f"Evidence chunks: {len(candidate['evidence_chunks'])}")

    print("\nEvidence:")
    for evidence in candidate["evidence_chunks"]:
        print(
            f"  Chunk {evidence['chunk_id']} "
            f"| score={evidence['similarity_score']:.4f}"
        )
        print(
            f"  {evidence['text'][:250].replace(chr(10), ' ')}"
        )

In [ ]:
import importlib.util

# Check which Google Generative AI SDK is currently available.
google_genai_available = importlib.util.find_spec("google.genai") is not None
google_generativeai_available = (
    importlib.util.find_spec("google.generativeai") is not None
)

print("=" * 60)
print("GEMINI SDK CHECK")
print("=" * 60)

print(f"google.genai available: {google_genai_available}")
print(f"google.generativeai available: {google_generativeai_available}")

In [ ]:
import os
from google import genai
from google.colab import userdata

# Load the Gemini API key securely from Google Colab Secrets.
api_key = userdata.get("GEMINI_API_KEY")

if not api_key:
    raise ValueError(
        "GEMINI_API_KEY was not found in Colab Secrets."
    )

# Initialize the current Google GenAI client.
gemini_client = genai.Client(
    api_key=api_key
)

print("=" * 60)
print("GEMINI CLIENT INITIALIZATION")
print("=" * 60)
print("Gemini API key loaded successfully from Colab Secrets.")
print("Google GenAI client initialized successfully.")

In [ ]:
# Test the current Gemini Interactions API with Gemini 3.6 Flash.
interaction = gemini_client.interactions.create(
    model="gemini-3.6-flash",
    input="Respond with exactly: Gemini connection successful."
)

print("=" * 60)
print("GEMINI API TEST")
print("=" * 60)
print(interaction.output_text)

In [ ]:
import json

# Define the recruiter query used for the RAG evaluation.
recruiter_query = "Junior Data Analyst with SQL and Tableau experience"


def build_candidate_evaluation_prompt(recruiter_query, candidate_context):
    """
    Build a grounded evaluation prompt for the top retrieved candidates.

    The LLM must use only the supplied resume evidence and must not
    invent skills, experience, qualifications, or demographic attributes.
    """

    candidates_json = json.dumps(
        candidate_context,
        ensure_ascii=False,
        indent=2
    )

    prompt = f"""
You are an AI recruitment assistant evaluating candidates retrieved
from a resume search engine.

Recruiter query:
{recruiter_query}

Your task is to evaluate the retrieved candidates using ONLY the
resume information provided below.

Candidate data:
{candidates_json}

Rules:
1. Do not invent or infer any information that is not explicitly present.
2. Do not infer age, gender, race, religion, nationality, disability,
   marital status, or any other protected or demographic attribute.
3. Do not use contact information such as email addresses.
4. If a requested skill or qualification is not explicitly supported,
   write "Not specified".
5. Distinguish between direct evidence and weak or indirect evidence.
6. Explain why each candidate is relevant to the recruiter query.
7. Mention specific skills, roles, tools, or experience only when
   supported by the provided resume evidence.
8. Identify important gaps between the recruiter query and the resume.
9. Perform a bias check based only on the available evidence.
10. Do not make hiring decisions or claims about who should be hired.
11. Return valid JSON only.

Required JSON structure:
{{
  "query": "{recruiter_query}",
  "candidates": [
    {{
      "rank": 1,
      "candidate_id": 0,
      "fit_summary": "Short evidence-based explanation.",
      "matching_evidence": [
        "Evidence explicitly supported by the resume."
      ],
      "gaps": [
        "Requirements that are not explicitly supported."
      ],
      "bias_check": "State whether the evaluation uses only job-relevant evidence."
    }}
  ]
}}
"""

    return prompt


evaluation_prompt = build_candidate_evaluation_prompt(
    recruiter_query=recruiter_query,
    candidate_context=final_top_3_context
)

print("=" * 60)
print("LLM EVALUATION PROMPT")
print("=" * 60)

print(f"Recruiter query: {recruiter_query}")
print(f"Candidates included: {len(final_top_3_context)}")
print(f"Prompt characters: {len(evaluation_prompt)}")

print("\nPrompt preview:")
print(evaluation_prompt[:3000])

In [ ]:
# Request a grounded candidate evaluation as plain text.
# JSON formatting is enforced through the prompt instead of the
# response_format parameter.

evaluation_prompt_text = evaluation_prompt + """

Return ONLY the JSON object requested above.
Do not return Markdown.
Do not return code fences.
Do not return an empty object.
"""

interaction = gemini_client.interactions.create(
    model="gemini-3.6-flash",
    input=evaluation_prompt_text
)

print("=" * 60)
print("GEMINI LLM EVALUATION")
print("=" * 60)

print(interaction.output_text)

In [ ]:
import re

# Keep only job-relevant entity types for the compressed LLM context.
RELEVANT_ENTITY_TYPES = {
    "Skills",
    "Designation",
    "Companies worked at",
    "Years of Experience",
    "Degree",
    "Graduation Year"
}


def select_relevant_entities(entities):
    """
    Keep only entities that can directly support candidate-job matching.
    Contact information and location are intentionally excluded.
    """
    return {
        key: values
        for key, values in entities.items()
        if key in RELEVANT_ENTITY_TYPES
    }


def build_compressed_llm_context(candidate_results, max_chunks_per_candidate=2):
    """
    Build a compact, evidence-focused context for Gemini.

    The full resume is intentionally excluded to reduce token usage.
    Only the highest-scoring retrieved chunks and job-relevant
    structured entities are provided to the LLM.
    """
    compressed_context = []

    for rank, candidate in enumerate(candidate_results[:3], start=1):
        sorted_chunks = sorted(
            candidate["matched_chunks"],
            key=lambda x: x["similarity_score"],
            reverse=True
        )

        selected_chunks = sorted_chunks[:max_chunks_per_candidate]

        evidence_chunks = []

        for chunk in selected_chunks:
            evidence_chunks.append({
                "chunk_id": chunk["chunk_id"],
                "similarity_score": round(
                    chunk["similarity_score"],
                    4
                ),
                "text": chunk["retrieval_text"]
            })

        compressed_context.append({
            "rank": rank,
            "candidate_id": candidate["candidate_id"],
            "retrieval_score": round(
                candidate["similarity_score"],
                4
            ),
            "entities": select_relevant_entities(
                candidate["entities"]
            ),
            "evidence_chunks": evidence_chunks
        })

    return compressed_context


compressed_llm_context = build_compressed_llm_context(
    deduplicated_candidates,
    max_chunks_per_candidate=2
)

print("=" * 60)
print("COMPRESSED LLM CONTEXT")
print("=" * 60)

print(f"Candidates included: {len(compressed_llm_context)}")
print(
    "Maximum evidence chunks per candidate: "
    f"{2}"
)

total_context_chars = 0

for candidate in compressed_llm_context:
    candidate_chars = sum(
        len(chunk["text"])
        for chunk in candidate["evidence_chunks"]
    )

    total_context_chars += candidate_chars

    print(
        f"\nCandidate {candidate['candidate_id']}: "
        f"{candidate_chars} evidence characters"
    )

print(f"\nTotal evidence characters: {total_context_chars:,}")

In [ ]:
import json

# Build a compact, evidence-grounded prompt for Gemini.
compressed_candidates_json = json.dumps(
    compressed_llm_context,
    ensure_ascii=False,
    indent=2
)

compressed_evaluation_prompt = f"""
You are an AI recruitment assistant evaluating candidates retrieved
from a resume search engine.

Recruiter query:
{recruiter_query}

Evaluate the candidates using ONLY the evidence provided below.

Candidate evidence:
{compressed_candidates_json}

Rules:
1. Use only information explicitly supported by the provided evidence.
2. Do not invent or infer skills, experience, qualifications, or roles.
3. Do not infer age, gender, race, religion, nationality, disability,
   marital status, or any other demographic or protected attribute.
4. Do not use contact information.
5. If a requested skill or qualification is not explicitly supported,
   write "Not specified".
6. Clearly distinguish direct evidence from missing evidence.
7. Identify both matching evidence and important gaps.
8. Consider the requested seniority level when explicitly supported
   by the evidence.
9. Do not make a hiring decision or recommend that a candidate be hired.
10. Bias check must be based only on job-relevant evidence.
11. Return ONLY valid JSON.
12. Do not return Markdown or code fences.

Return exactly this JSON structure:

{{
  "query": "{recruiter_query}",
  "candidates": [
    {{
      "rank": 1,
      "candidate_id": 0,
      "fit_summary": "Evidence-based summary.",
      "matching_evidence": [
        "Explicit evidence supporting relevance."
      ],
      "gaps": [
        "Important missing or unsupported requirements."
      ],
      "bias_check": "Short evidence-based bias check."
    }}
  ]
}}
"""

print("=" * 60)
print("COMPRESSED GEMINI PROMPT")
print("=" * 60)

print(f"Prompt characters: {len(compressed_evaluation_prompt):,}")
print(
    f"Previous prompt characters: "
    f"{len(evaluation_prompt):,}"
)
print(
    f"Reduction: "
    f"{(1 - len(compressed_evaluation_prompt) / len(evaluation_prompt)) * 100:.2f}%"
)

In [ ]:
# Run Gemini using the compressed, evidence-focused context.
compressed_interaction = gemini_client.interactions.create(
    model="gemini-3.6-flash",
    input=compressed_evaluation_prompt
)

print("=" * 60)
print("COMPRESSED GEMINI EVALUATION")
print("=" * 60)

print(compressed_interaction.output_text)

In [ ]:
# Test the retrieval pipeline with multiple realistic recruiter queries.
# Candidate profiles are attached before duplicate detection because
# duplicate comparison requires the original resume text.

test_queries = [
    "Junior Data Analyst with SQL and Tableau experience",
    "Python developer with machine learning experience",
    "NLP engineer with Python and NLP experience",
    "Data analyst with SQL and Power BI experience",
    "Software engineer with cloud and AWS experience"
]

print("=" * 60)
print("MULTI-QUERY RETRIEVAL TEST")
print("=" * 60)

for query_number, query in enumerate(test_queries, start=1):
    results = retrieve_candidates(
        query=query,
        top_k_chunks=30,
        top_k_candidates=5
    )

    # Attach the original candidate profile required for duplicate comparison.
    for candidate in results:
        profile = get_candidate_profile(
            candidate["candidate_id"]
        )

        candidate["resume_text"] = profile["text"]
        candidate["entities"] = profile["entities"]

    results = deduplicate_candidates(
        results,
        similarity_threshold=DUPLICATE_THRESHOLD
    )

    print(f"\nQuery {query_number}: {query}")
    print("-" * 60)

    for rank, candidate in enumerate(results[:5], start=1):
        print(
            f"{rank}. Candidate {candidate['candidate_id']} "
            f"| Score: {candidate['similarity_score']:.4f}"
        )

In [ ]:
# Inspect the actual evidence retrieved for each recruiter query.
# This allows us to verify semantic retrieval using real resume text
# instead of relying only on similarity scores.

for query_number, query in enumerate(test_queries, start=1):

    results = retrieve_candidates(
        query=query,
        top_k_chunks=30,
        top_k_candidates=5
    )

    # Attach candidate profiles for duplicate detection.
    for candidate in results:
        profile = get_candidate_profile(
            candidate["candidate_id"]
        )

        candidate["resume_text"] = profile["text"]
        candidate["entities"] = profile["entities"]

    results = deduplicate_candidates(
        results,
        similarity_threshold=DUPLICATE_THRESHOLD
    )

    print("=" * 80)
    print(f"QUERY {query_number}: {query}")
    print("=" * 80)

    for rank, candidate in enumerate(results[:3], start=1):

        print(
            f"\nCandidate {candidate['candidate_id']} "
            f"| Rank: {rank} "
            f"| Score: {candidate['similarity_score']:.4f}"
        )

        matched_chunks = sorted(
            candidate["matched_chunks"],
            key=lambda x: x["similarity_score"],
            reverse=True
        )

        for chunk in matched_chunks[:2]:
            print(
                f"\nChunk {chunk['chunk_id']} "
                f"| Score: {chunk['similarity_score']:.4f}"
            )

            print(chunk["text"][:800])

In [ ]:
import re

# Build a vocabulary of technical skills and tools that actually
# appear in the resume dataset.
skill_vocabulary = set()

for resume in resume_records:
    skills = resume["entities"].get("Skills", [])

    for skill in skills:
        skill = skill.strip()

        if skill:
            skill_vocabulary.add(skill.lower())

# Add common multi-word technical terms that may appear in resumes
# but may not always be consistently annotated.
known_terms = {
    "machine learning",
    "deep learning",
    "natural language processing",
    "nlp",
    "computer vision",
    "data analysis",
    "data science",
    "python",
    "sql",
    "tableau",
    "power bi",
    "aws",
    "azure",
    "tensorflow",
    "pytorch",
    "scikit-learn",
    "pandas",
    "numpy"
}

skill_vocabulary.update(known_terms)

print("=" * 60)
print("SKILL VOCABULARY")
print("=" * 60)

print(f"Unique dataset skills: {len(skill_vocabulary)}")

sample_skills = sorted(skill_vocabulary)[:50]

print("\nSample skills:")
for skill in sample_skills:
    print(f"- {skill}")

In [ ]:
# Define a controlled vocabulary of technical requirements.
# These terms are intentionally limited to job-relevant technologies,
# frameworks, tools, and AI/ML concepts commonly found in the dataset.

TECHNICAL_REQUIREMENTS = [
    "python",
    "java",
    "c",
    "c++",
    "c#",
    "sql",
    "pl/sql",
    "mysql",
    "oracle",
    "sql server",
    "tableau",
    "power bi",
    "excel",
    "aws",
    "azure",
    "gcp",
    "docker",
    "kubernetes",
    "jenkins",
    "git",
    "tensorflow",
    "pytorch",
    "scikit-learn",
    "pandas",
    "numpy",
    "machine learning",
    "deep learning",
    "natural language processing",
    "nlp",
    "computer vision",
    "data science",
    "data analysis",
    "data analytics",
    "business intelligence",
    "big data",
    "hadoop",
    "spark",
    "hive",
    "etl",
    "ssas",
    "ssis",
    "mongodb",
    "redis",
    "flask",
    "django",
    "fastapi",
    "transformers",
    "bert",
    "llm",
    "generative ai",
    "langchain"
]

TECHNICAL_REQUIREMENTS = sorted(
    set(TECHNICAL_REQUIREMENTS),
    key=len,
    reverse=True
)

print("=" * 60)
print("CONTROLLED TECHNICAL REQUIREMENT VOCABULARY")
print("=" * 60)

print(f"Total controlled requirements: {len(TECHNICAL_REQUIREMENTS)}")

print("\nRequirements:")
for requirement in TECHNICAL_REQUIREMENTS:
    print(f"- {requirement}")

In [ ]:
# Extract technical requirements explicitly mentioned in a recruiter query.
# Longer phrases are checked first to avoid partial matches such as
# "sql" being detected inside "sql server" or "c" inside "c++".

def extract_requirements(query):
    query_normalized = re.sub(
        r"\s+",
        " ",
        query.lower().strip()
    )

    detected_requirements = []

    for requirement in TECHNICAL_REQUIREMENTS:
        # Escape the requirement so special characters such as
        # "+" and "#" are treated literally.
        escaped_requirement = re.escape(requirement)

        # Use boundaries where possible to prevent partial matches.
        pattern = rf"(?<![a-z0-9+#]){escaped_requirement}(?![a-z0-9+#])"

        if re.search(pattern, query_normalized):
            detected_requirements.append(requirement)

    return detected_requirements


test_queries = [
    "Junior Data Analyst with SQL and Tableau experience",
    "Python developer with machine learning experience",
    "NLP engineer with Python and NLP experience",
    "Data analyst with SQL and Power BI experience",
    "Software engineer with cloud and AWS experience"
]

print("=" * 60)
print("REQUIREMENT EXTRACTION TEST")
print("=" * 60)

for query in test_queries:
    requirements = extract_requirements(query)

    print(f"\nQuery: {query}")
    print(f"Detected requirements: {requirements}")

In [ ]:
# Define safe aliases for requirements that may appear in
# different forms inside resumes.
REQUIREMENT_ALIASES = {
    "nlp": [
        "nlp",
        "natural language processing"
    ],
    "natural language processing": [
        "natural language processing",
        "nlp"
    ],
    "machine learning": [
        "machine learning",
        "machine-learning"
    ],
    "deep learning": [
        "deep learning",
        "deep-learning"
    ],
    "computer vision": [
        "computer vision",
        "computer-vision"
    ],
    "power bi": [
        "power bi",
        "powerbi"
    ],
    "scikit-learn": [
        "scikit-learn",
        "sklearn"
    ],
    "sql server": [
        "sql server",
        "microsoft sql server"
    ],
    "pl/sql": [
        "pl/sql",
        "plsql"
    ]
}


def requirement_in_text(requirement, text):
    # Normalize whitespace and case before matching.
    text_normalized = re.sub(
        r"\s+",
        " ",
        text.lower()
    )

    aliases = REQUIREMENT_ALIASES.get(
        requirement,
        [requirement]
    )

    for alias in aliases:
        escaped_alias = re.escape(alias)

        pattern = rf"(?<![a-z0-9+#]){escaped_alias}(?![a-z0-9+#])"

        if re.search(pattern, text_normalized):
            return True

    return False


def match_candidate_requirements(candidate, requirements):
    # Use the cleaned full resume text for requirement matching.
    resume_text = clean_resume_text(
        candidate["resume_text"]
    )

    matched_requirements = []
    missing_requirements = []

    for requirement in requirements:
        if requirement_in_text(
            requirement,
            resume_text
        ):
            matched_requirements.append(requirement)
        else:
            missing_requirements.append(requirement)

    total_requirements = len(requirements)

    if total_requirements > 0:
        coverage = (
            len(matched_requirements)
            / total_requirements
        )
    else:
        coverage = 0.0

    return {
        "matched_requirements": matched_requirements,
        "missing_requirements": missing_requirements,
        "requirement_coverage": coverage
    }


print("=" * 60)
print("REQUIREMENT MATCHING FUNCTION READY")
print("=" * 60)

print("Function: match_candidate_requirements")
print("Requirement matching uses cleaned resume text.")
print("No candidate ranking has been changed yet.")

In [ ]:
# Evaluate explicit requirement coverage for the candidates
# retrieved by the existing FAISS semantic search pipeline.

print("=" * 80)
print("FAISS RESULTS + EXPLICIT REQUIREMENT COVERAGE")
print("=" * 80)

for query in test_queries:
    requirements = extract_requirements(query)

    # Retrieve candidates using the existing semantic pipeline.
    retrieved_candidates = retrieve_candidates(
        query=query,
        top_k_chunks=30,
        top_k_candidates=5
    )

    # Attach the complete candidate profile before deduplication.
    for candidate in retrieved_candidates:
        profile = get_candidate_profile(
            candidate["candidate_id"]
        )

        candidate["resume_text"] = profile["text"]
        candidate["entities"] = profile["entities"]

    # Remove near-duplicate candidate profiles.
    retrieved_candidates = deduplicate_candidates(
        retrieved_candidates,
        similarity_threshold=DUPLICATE_THRESHOLD
    )

    print("\n" + "-" * 80)
    print(f"QUERY: {query}")
    print(f"REQUIREMENTS: {requirements}")
    print("-" * 80)

    for rank, candidate in enumerate(
        retrieved_candidates,
        start=1
    ):
        requirement_result = match_candidate_requirements(
            candidate,
            requirements
        )

        print(
            f"\nFAISS Rank: {rank}"
            f" | Candidate: {candidate['candidate_id']}"
            f" | Similarity: {candidate['similarity_score']:.4f}"
        )

        print(
            "Matched:"
            f" {requirement_result['matched_requirements']}"
        )

        print(
            "Missing:"
            f" {requirement_result['missing_requirements']}"
        )

        print(
            "Coverage:"
            f" {requirement_result['requirement_coverage']:.2%}"
        )

In [ ]:
# Validate the requirement matching logic across the full resume dataset.
# This checks how many candidates explicitly mention each requirement.

print("=" * 80)
print("REQUIREMENT MATCHING VALIDATION")
print("=" * 80)

validation_requirements = [
    "python",
    "sql",
    "tableau",
    "power bi",
    "aws",
    "machine learning",
    "nlp",
    "tensorflow",
    "pytorch"
]

for requirement in validation_requirements:
    matched_candidates = []

    for resume in resume_records:
        resume_text = clean_resume_text(
            resume["text"]
        )

        if requirement_in_text(
            requirement,
            resume_text
        ):
            matched_candidates.append(
                resume["candidate_id"]
            )

    print(
        f"\n{requirement}: "
        f"{len(matched_candidates)} / "
        f"{len(resume_records)} candidates"
    )

    print(
        "Sample candidate IDs:",
        matched_candidates[:15]
    )

In [ ]:
# Combine semantic similarity with explicit requirement coverage.
# Semantic similarity captures overall query-resume relevance,
# while requirement coverage rewards candidates who explicitly
# satisfy the technical requirements mentioned in the query.

SEMANTIC_WEIGHT = 0.70
REQUIREMENT_WEIGHT = 0.30


def calculate_hybrid_score(
    semantic_score,
    requirement_coverage
):
    return (
        SEMANTIC_WEIGHT * semantic_score
        + REQUIREMENT_WEIGHT * requirement_coverage
    )


def rerank_candidates(candidate_results, query):
    # Extract explicit technical requirements from the recruiter query.
    requirements = extract_requirements(query)

    reranked_candidates = []

    for candidate in candidate_results:
        requirement_result = match_candidate_requirements(
            candidate,
            requirements
        )

        semantic_score = candidate["similarity_score"]
        requirement_coverage = (
            requirement_result["requirement_coverage"]
        )

        hybrid_score = calculate_hybrid_score(
            semantic_score,
            requirement_coverage
        )

        reranked_candidate = candidate.copy()

        reranked_candidate.update({
            "matched_requirements":
                requirement_result["matched_requirements"],
            "missing_requirements":
                requirement_result["missing_requirements"],
            "requirement_coverage":
                requirement_coverage,
            "hybrid_score":
                hybrid_score
        })

        reranked_candidates.append(
            reranked_candidate
        )

    # Rank candidates using the hybrid score.
    reranked_candidates.sort(
        key=lambda x: x["hybrid_score"],
        reverse=True
    )

    return reranked_candidates


print("=" * 80)
print("HYBRID RERANKER READY")
print("=" * 80)

print(f"Semantic weight: {SEMANTIC_WEIGHT:.2f}")
print(f"Requirement weight: {REQUIREMENT_WEIGHT:.2f}")
print("Scoring formula:")
print(
    "Hybrid Score = "
    f"{SEMANTIC_WEIGHT:.2f} * Semantic Similarity + "
    f"{REQUIREMENT_WEIGHT:.2f} * Requirement Coverage"
)

In [ ]:
# Compare the original FAISS ranking with the new hybrid ranking.
# The comparison uses the same five test queries and the same
# retrieved candidate pool to isolate the effect of reranking.

print("=" * 100)
print("FAISS vs HYBRID RANKING COMPARISON")
print("=" * 100)

for query in test_queries:
    # Retrieve candidates using the existing FAISS pipeline.
    retrieved_candidates = retrieve_candidates(
        query=query,
        top_k_chunks=30,
        top_k_candidates=5
    )

    # Attach complete candidate profiles before deduplication.
    for candidate in retrieved_candidates:
        profile = get_candidate_profile(
            candidate["candidate_id"]
        )

        candidate["resume_text"] = profile["text"]
        candidate["entities"] = profile["entities"]

    # Remove near-duplicate candidates.
    retrieved_candidates = deduplicate_candidates(
        retrieved_candidates,
        similarity_threshold=DUPLICATE_THRESHOLD
    )

    # Apply hybrid reranking to the same candidate pool.
    reranked_candidates = rerank_candidates(
        retrieved_candidates,
        query=query
    )

    requirements = extract_requirements(query)

    print("\n" + "-" * 100)
    print(f"QUERY: {query}")
    print(f"REQUIREMENTS: {requirements}")
    print("-" * 100)

    print("\nOriginal FAISS ranking:")

    for rank, candidate in enumerate(
        retrieved_candidates,
        start=1
    ):
        requirement_result = match_candidate_requirements(
            candidate,
            requirements
        )

        print(
            f"{rank}. Candidate {candidate['candidate_id']} "
            f"| Similarity: {candidate['similarity_score']:.4f} "
            f"| Coverage: "
            f"{requirement_result['requirement_coverage']:.0%}"
        )

    print("\nHybrid ranking:")

    for rank, candidate in enumerate(
        reranked_candidates,
        start=1
    ):
        print(
            f"{rank}. Candidate {candidate['candidate_id']} "
            f"| Similarity: {candidate['similarity_score']:.4f} "
            f"| Coverage: {candidate['requirement_coverage']:.0%} "
            f"| Hybrid: {candidate['hybrid_score']:.4f} "
            f"| Matched: {candidate['matched_requirements']} "
            f"| Missing: {candidate['missing_requirements']}"
        )

In [ ]:
def search_talent(
    query,
    top_k_chunks=30,
    top_k_candidates=5,
    final_top_k=3
):
    """
    Run the complete talent retrieval pipeline.

    Pipeline:
    1. Extract explicit technical requirements.
    2. Retrieve relevant resume chunks using FAISS.
    3. Group retrieved chunks by candidate.
    4. Attach candidate profiles.
    5. Remove near-duplicate resumes.
    6. Match explicit requirements.
    7. Apply hybrid reranking.
    8. Return the final top candidates.
    """

    # Extract explicit technical requirements from the recruiter query.
    requirements = extract_requirements(query)

    # Retrieve candidate chunks using semantic search.
    retrieved_candidates = retrieve_candidates(
        query=query,
        top_k_chunks=top_k_chunks,
        top_k_candidates=top_k_candidates
    )

    # Attach the complete candidate profile to each result.
    for candidate in retrieved_candidates:
        profile = get_candidate_profile(
            candidate["candidate_id"]
        )

        candidate["resume_text"] = profile["text"]
        candidate["entities"] = profile["entities"]

    # Remove near-duplicate candidate resumes.
    retrieved_candidates = deduplicate_candidates(
        retrieved_candidates,
        similarity_threshold=DUPLICATE_THRESHOLD
    )

    # Apply requirement-aware hybrid reranking.
    reranked_candidates = rerank_candidates(
        retrieved_candidates,
        query=query
    )

    # Keep only the final top candidates.
    final_candidates = reranked_candidates[:final_top_k]

    # Assign the final ranking.
    for rank, candidate in enumerate(
        final_candidates,
        start=1
    ):
        candidate["final_rank"] = rank

    return {
        "query": query,
        "requirements": requirements,
        "candidates": final_candidates
    }


print("=" * 60)
print("TALENT SEARCH PIPELINE")
print("=" * 60)

test_query = "Junior Data Analyst with SQL and Tableau experience"

search_results = search_talent(
    query=test_query,
    top_k_chunks=30,
    top_k_candidates=5,
    final_top_k=3
)

print(f"Query: {search_results['query']}")
print(f"Requirements: {search_results['requirements']}")
print(f"Final candidates: {len(search_results['candidates'])}")

print("\nFinal ranking:")

for candidate in search_results["candidates"]:
    print(
        f"{candidate['final_rank']}. "
        f"Candidate {candidate['candidate_id']} | "
        f"Hybrid: {candidate['hybrid_score']:.4f} | "
        f"Similarity: {candidate['similarity_score']:.4f} | "
        f"Coverage: {candidate['requirement_coverage']:.0%} | "
        f"Matched: {candidate['matched_requirements']} | "
        f"Missing: {candidate['missing_requirements']}"
    )

In [ ]:
def build_gemini_context(search_results, max_chunks_per_candidate=2):
    """
    Build a compact evidence context for Gemini.

    Only job-relevant entities and the strongest retrieved
    evidence chunks are included.
    """

    gemini_context = []

    for candidate in search_results["candidates"]:
        # Sort retrieved chunks by semantic similarity.
        sorted_chunks = sorted(
            candidate["matched_chunks"],
            key=lambda x: x["similarity_score"],
            reverse=True
        )

        # Keep only the strongest evidence chunks.
        selected_chunks = sorted_chunks[
            :max_chunks_per_candidate
        ]

        evidence_chunks = []

        for chunk in selected_chunks:
            evidence_chunks.append({
                "chunk_id": chunk["chunk_id"],
                "similarity_score": round(
                    chunk["similarity_score"],
                    4
                ),
                "text": chunk["retrieval_text"]
            })

        gemini_context.append({
            "rank": candidate["final_rank"],
            "candidate_id": candidate["candidate_id"],
            "semantic_score": round(
                candidate["similarity_score"],
                4
            ),
            "hybrid_score": round(
                candidate["hybrid_score"],
                4
            ),
            "requirement_coverage": round(
                candidate["requirement_coverage"],
                4
            ),
            "matched_requirements":
                candidate["matched_requirements"],
            "missing_requirements":
                candidate["missing_requirements"],
            "entities": select_relevant_entities(
                candidate["entities"]
            ),
            "evidence_chunks": evidence_chunks
        })

    return gemini_context


gemini_context = build_gemini_context(
    search_results,
    max_chunks_per_candidate=2
)

gemini_context_json = json.dumps(
    gemini_context,
    ensure_ascii=False,
    indent=2
)

print("=" * 60)
print("GEMINI CONTEXT")
print("=" * 60)

print(f"Candidates included: {len(gemini_context)}")
print(
    f"Context size: "
    f"{len(gemini_context_json):,} characters"
)

for candidate in gemini_context:
    evidence_length = sum(
        len(chunk["text"])
        for chunk in candidate["evidence_chunks"]
    )

    print(
        f"Candidate {candidate['candidate_id']} | "
        f"Evidence characters: {evidence_length:,}"
    )

In [ ]:
recruiter_query = search_results["query"]

gemini_prompt = f"""
You are an AI recruitment assistant working inside a
Retrieval-Augmented Generation (RAG) talent search system.

Recruiter query:
{recruiter_query}

The retrieval system selected the following candidates.
The provided information is the only evidence you are allowed to use.

Candidate evidence:
{gemini_context_json}

Your task is to evaluate how each candidate matches the
recruiter's job-related requirements.

Evaluation rules:

1. Use ONLY the candidate evidence provided above.
2. Do not invent skills, experience, qualifications, job titles,
   education, or achievements.
3. Do not infer information that is not explicitly stated.
4. If a requested requirement is not supported by the evidence,
   state "Not specified".
5. Clearly identify direct matching evidence.
6. Clearly identify important gaps or missing requirements.
7. Consider seniority only when it is explicitly supported by
   the provided evidence.
8. Do not use email addresses or other contact information.
9. Do not infer or use age, gender, race, religion, nationality,
   disability, marital status, or other protected characteristics.
10. Do not make a hiring decision.
11. Do not recommend hiring or rejecting any candidate.
12. The bias check must confirm that the evaluation relies only
    on job-relevant evidence.
13. Do not treat semantic similarity alone as proof that a candidate
    has a required skill.
14. Return ONLY valid JSON.
15. Do not use Markdown or code fences.

Return exactly this structure:

{{
  "query": "{recruiter_query}",
  "candidates": [
    {{
      "rank": 1,
      "candidate_id": 0,
      "fit_summary": "Concise evidence-based summary.",
      "matching_evidence": [
        "Direct evidence supporting the query."
      ],
      "gaps": [
        "Missing or unsupported requirement."
      ],
      "bias_check": "Evaluation uses only job-relevant evidence."
    }}
  ]
}}
"""

print("=" * 60)
print("GEMINI EVALUATION PROMPT")
print("=" * 60)

print(f"Query: {recruiter_query}")
print(
    f"Prompt size: "
    f"{len(gemini_prompt):,} characters"
)

print("\nPrompt prepared successfully.")

In [ ]:
# Send the final grounded evaluation request to Gemini.
# This cell makes exactly one Gemini API request.

evaluation_interaction = gemini_client.interactions.create(
    model="gemini-3.6-flash",
    input=gemini_prompt
)

gemini_raw_output = evaluation_interaction.output_text

print("=" * 60)
print("GEMINI EVALUATION RESPONSE")
print("=" * 60)

print(gemini_raw_output)

In [ ]:
import json

# Parse the Gemini response into a Python dictionary.
# No Gemini API request is made in this cell.

try:
    gemini_evaluation = json.loads(
        gemini_raw_output
    )
except json.JSONDecodeError as error:
    raise ValueError(
        f"Gemini returned invalid JSON: {error}"
    )

print("=" * 60)
print("GEMINI JSON VALIDATION")
print("=" * 60)

print(
    f"Query: {gemini_evaluation['query']}"
)

print(
    f"Candidates returned: "
    f"{len(gemini_evaluation['candidates'])}"
)

required_candidate_fields = {
    "rank",
    "candidate_id",
    "fit_summary",
    "matching_evidence",
    "gaps",
    "bias_check"
}

validation_passed = True

for candidate in gemini_evaluation["candidates"]:
    missing_fields = (
        required_candidate_fields
        - set(candidate.keys())
    )

    if missing_fields:
        validation_passed = False

        print(
            f"Candidate {candidate.get('candidate_id')} "
            f"missing fields: {missing_fields}"
        )

if validation_passed:
    print("\nJSON structure validation PASSED.")
else:
    print("\nJSON structure validation FAILED.")

In [ ]:
from pathlib import Path
import json
import hashlib

# Create a persistent cache directory for Gemini evaluations.
CACHE_DIR = Path("/content/gemini_cache")
CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


def create_gemini_cache_key(
    query,
    candidate_results
):
    """
    Create a deterministic cache key from the query
    and the retrieved candidate ranking.
    """

    cache_payload = {
        "query": query,
        "candidates": [
            {
                "candidate_id": candidate["candidate_id"],
                "hybrid_score": round(
                    candidate["hybrid_score"],
                    6
                ),
                "matched_requirements":
                    candidate["matched_requirements"],
                "missing_requirements":
                    candidate["missing_requirements"]
            }
            for candidate in candidate_results[:3]
        ]
    }

    serialized_payload = json.dumps(
        cache_payload,
        sort_keys=True,
        ensure_ascii=False
    )

    return hashlib.sha256(
        serialized_payload.encode("utf-8")
    ).hexdigest()


def save_gemini_evaluation(
    cache_key,
    evaluation
):
    """
    Save a validated Gemini evaluation to the local cache.
    """

    cache_path = CACHE_DIR / f"{cache_key}.json"

    with open(
        cache_path,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            evaluation,
            f,
            ensure_ascii=False,
            indent=2
        )

    return cache_path


# Generate a cache key for the current search result.
cache_key = create_gemini_cache_key(
    search_results["query"],
    search_results["candidates"]
)

# Save the validated Gemini evaluation.
cache_path = save_gemini_evaluation(
    cache_key,
    gemini_evaluation
)

print("=" * 60)
print("GEMINI EVALUATION CACHED")
print("=" * 60)

print(f"Cache key: {cache_key}")
print(f"Cache file: {cache_path}")
print("Evaluation saved successfully.")

In [ ]:
def run_talent_search(
    query,
    top_k_chunks=30,
    top_k_candidates=5,
    final_top_k=3
):
    """
    Run the complete RAG talent search pipeline.

    The pipeline performs:
    1. Semantic retrieval with FAISS.
    2. Candidate-level grouping.
    3. Duplicate removal.
    4. Requirement matching.
    5. Hybrid reranking.
    6. Gemini evaluation for the final candidates.
    7. Cache lookup to avoid repeated Gemini requests.
    """

    # Run the local retrieval and hybrid ranking pipeline.
    search_results = search_talent(
        query=query,
        top_k_chunks=top_k_chunks,
        top_k_candidates=top_k_candidates,
        final_top_k=final_top_k
    )

    final_candidates = search_results["candidates"]

    # Create a deterministic cache key from the query and ranking.
    cache_key = create_gemini_cache_key(
        search_results["query"],
        final_candidates
    )

    cache_path = CACHE_DIR / f"{cache_key}.json"

    # Check whether a Gemini evaluation already exists.
    if cache_path.exists():

        with open(
            cache_path,
            "r",
            encoding="utf-8"
        ) as f:
            gemini_evaluation = json.load(f)

        source = "cache"

    else:

        # Build the compact evidence context for Gemini.
        gemini_context = build_gemini_context(
            search_results,
            max_chunks_per_candidate=2
        )

        gemini_context_json = json.dumps(
            gemini_context,
            ensure_ascii=False,
            indent=2
        )

        # Build the grounded Gemini prompt.
        recruiter_query = search_results["query"]

        evaluation_prompt = f"""
You are an AI recruitment assistant working inside a
Retrieval-Augmented Generation (RAG) talent search system.

Recruiter query:
{recruiter_query}

The retrieval system selected the following candidates.
The provided information is the only evidence you are allowed to use.

Candidate evidence:
{gemini_context_json}

Evaluation rules:

1. Use ONLY the candidate evidence provided above.
2. Do not invent skills, experience, qualifications, job titles,
   education, or achievements.
3. Do not infer information that is not explicitly stated.
4. If a requested requirement is not supported by the evidence,
   state "Not specified".
5. Clearly identify direct matching evidence.
6. Clearly identify important gaps or missing requirements.
7. Consider seniority only when it is explicitly supported.
8. Do not use contact information.
9. Do not infer or use demographic or protected characteristics.
10. Do not make a hiring decision.
11. Do not recommend hiring or rejecting any candidate.
12. Do not treat semantic similarity alone as proof of a skill.
13. Return ONLY valid JSON.
14. Do not return Markdown or code fences.

Return exactly this structure:

{{
  "query": "{recruiter_query}",
  "candidates": [
    {{
      "rank": 1,
      "candidate_id": 0,
      "fit_summary": "Concise evidence-based summary.",
      "matching_evidence": [
        "Direct evidence supporting the query."
      ],
      "gaps": [
        "Missing or unsupported requirement."
      ],
      "bias_check": "Evaluation uses only job-relevant evidence."
    }}
  ]
}}
"""

        # Make one Gemini request only when no cached result exists.
        interaction = gemini_client.interactions.create(
            model="gemini-3.6-flash",
            input=evaluation_prompt
        )

        raw_output = interaction.output_text

        # Parse the Gemini response.
        try:
            gemini_evaluation = json.loads(
                raw_output
            )
        except json.JSONDecodeError as error:
            raise ValueError(
                f"Gemini returned invalid JSON: {error}"
            )

        # Save the validated evaluation for future reuse.
        save_gemini_evaluation(
            cache_key,
            gemini_evaluation
        )

        source = "gemini"

    return {
        "query": search_results["query"],
        "requirements": search_results["requirements"],
        "retrieved_candidates": final_candidates,
        "gemini_evaluation": gemini_evaluation,
        "evaluation_source": source
    }


print("=" * 60)
print("COMPLETE TALENT SEARCH PIPELINE")
print("=" * 60)

print("Pipeline function created successfully.")
print("Gemini cache protection is enabled.")

In [ ]:
test_query = "Junior Data Analyst with SQL and Tableau experience"

final_result = run_talent_search(
    query=test_query,
    top_k_chunks=30,
    top_k_candidates=5,
    final_top_k=3
)

print("=" * 60)
print("FINAL TALENT SEARCH TEST")
print("=" * 60)

print(f"Query: {final_result['query']}")
print(f"Requirements: {final_result['requirements']}")
print(
    f"Evaluation source: "
    f"{final_result['evaluation_source']}"
)

print(
    f"Candidates returned: "
    f"{len(final_result['retrieved_candidates'])}"
)

print("\nFinal candidates:")

for candidate in final_result["retrieved_candidates"]:
    print(
        f"{candidate['final_rank']}. "
        f"Candidate {candidate['candidate_id']} | "
        f"Hybrid: {candidate['hybrid_score']:.4f} | "
        f"Coverage: "
        f"{candidate['requirement_coverage']:.0%}"
    )

print("\nGemini evaluation:")

for candidate in final_result["gemini_evaluation"]["candidates"]:
    print(
        f"{candidate['rank']}. "
        f"Candidate {candidate['candidate_id']}"
    )
    print(
        f"   Summary: "
        f"{candidate['fit_summary']}"
    )

In [ ]:
import os
import json
import pickle
import faiss

# Define the project artifact directory.
ARTIFACT_DIR = "/content/rag_talent_search_artifacts"

os.makedirs(
    ARTIFACT_DIR,
    exist_ok=True
)

# Save the FAISS vector index.
faiss_index_path = os.path.join(
    ARTIFACT_DIR,
    "resume_faiss.index"
)

faiss.write_index(
    faiss_index,
    faiss_index_path
)

print("=" * 60)
print("FAISS ARTIFACT SAVED")
print("=" * 60)

print(f"Path: {faiss_index_path}")
print(f"Vectors: {faiss_index.ntotal}")
print(f"Dimension: {faiss_index.d}")

In [ ]:
import os
import json

# Save the chunk metadata used to map FAISS vectors
# back to their original candidate and chunk information.

chunks_path = os.path.join(
    ARTIFACT_DIR,
    "chunks.json"
)

with open(
    chunks_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        chunks,
        f,
        ensure_ascii=False,
        indent=2
    )

print("=" * 60)
print("CHUNK METADATA SAVED")
print("=" * 60)

print(f"Path: {chunks_path}")
print(f"Chunks saved: {len(chunks)}")

if len(chunks) == faiss_index.ntotal:
    print("\nChunk-to-FAISS mapping validation PASSED.")
else:
    print("\nChunk-to-FAISS mapping validation FAILED.")

In [ ]:
import os
import json

# Save sanitized candidate profiles for the production application.
# Raw resume text and direct contact information are not stored.
sanitized_candidate_profiles = {}

for resume in resume_records:
    candidate_id = resume["candidate_id"]

    sanitized_candidate_profiles[str(candidate_id)] = {
        "candidate_id": candidate_id,
        "resume_text": clean_resume_text(
            resume["text"]
        ),
        "entities": select_relevant_entities(
            resume["entities"]
        )
    }

candidate_profiles_path = os.path.join(
    ARTIFACT_DIR,
    "candidate_profiles.json"
)

with open(
    candidate_profiles_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        sanitized_candidate_profiles,
        f,
        ensure_ascii=False,
        indent=2
    )

print("=" * 60)
print("SANITIZED CANDIDATE PROFILES SAVED")
print("=" * 60)

print(f"Path: {candidate_profiles_path}")
print(
    f"Candidates saved: "
    f"{len(sanitized_candidate_profiles)}"
)

# Validate that direct email fields were excluded.
email_fields_found = []

for candidate_id, profile in sanitized_candidate_profiles.items():
    entities = profile.get("entities", {})

    if "Email Address" in entities:
        email_fields_found.append(candidate_id)

if not email_fields_found:
    print("PII validation PASSED: no email fields stored.")
else:
    print(
        "PII validation FAILED: "
        f"{len(email_fields_found)} profiles contain email fields."
    )

In [ ]:
import os
import json

# Store all production pipeline settings in one configuration file.
# This allows the Streamlit application to reproduce the same
# retrieval and ranking behavior used during development.

project_config = {
    "project_name": "RAG-Powered Talent Search Engine",

    "embedding": {
        "model_name": embedding_model_name,
        "dimension": int(embeddings.shape[1]),
        "normalize_embeddings": True
    },

    "chunking": {
        "chunk_size": CHUNK_SIZE,
        "chunk_overlap": CHUNK_OVERLAP,
        "total_chunks": len(chunks)
    },

    "retrieval": {
        "default_top_k_chunks": 30,
        "default_top_k_candidates": 5,
        "default_final_top_k": 3,
        "faiss_index_type": "IndexFlatIP"
    },

    "deduplication": {
        "similarity_threshold": DUPLICATE_THRESHOLD,
        "comparison_method": "SequenceMatcher"
    },

    "reranking": {
        "semantic_weight": SEMANTIC_WEIGHT,
        "requirement_weight": REQUIREMENT_WEIGHT,
        "requirement_matching": "controlled technical vocabulary"
    },

    "dataset": {
        "candidate_count": len(resume_records),
        "chunk_count": len(chunks)
    },

    "llm_evaluation": {
        "provider": "Google Gemini",
        "model": "gemini-3.6-flash",
        "top_candidates_evaluated": 3,
        "output_format": "JSON",
        "cache_enabled": True
    }
}

config_path = os.path.join(
    ARTIFACT_DIR,
    "config.json"
)

with open(
    config_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        project_config,
        f,
        ensure_ascii=False,
        indent=2
    )

print("=" * 60)
print("PROJECT CONFIGURATION SAVED")
print("=" * 60)

print(f"Path: {config_path}")

print("\nConfiguration summary:")
print(f"Embedding model: {embedding_model_name}")
print(f"Embedding dimension: {embeddings.shape[1]}")
print(f"Chunk size: {CHUNK_SIZE}")
print(f"Chunk overlap: {CHUNK_OVERLAP}")
print(f"FAISS index: IndexFlatIP")
print(f"Semantic weight: {SEMANTIC_WEIGHT}")
print(f"Requirement weight: {REQUIREMENT_WEIGHT}")
print(f"Gemini model: gemini-3.6-flash")
print(f"Candidates: {len(resume_records)}")
print(f"Chunks: {len(chunks)}")

In [ ]:
import os
import json
import faiss

# Validate all saved production artifacts before building the application.
# This ensures that the FAISS index, chunk metadata, candidate profiles,
# and configuration file are internally consistent.

print("=" * 60)
print("PRODUCTION ARTIFACT VALIDATION")
print("=" * 60)

# ------------------------------------------------------------
# 1. Verify required files exist.
# ------------------------------------------------------------

required_files = [
    "resume_faiss.index",
    "chunks.json",
    "candidate_profiles.json",
    "config.json"
]

missing_files = []

for filename in required_files:
    file_path = os.path.join(
        ARTIFACT_DIR,
        filename
    )

    if not os.path.exists(file_path):
        missing_files.append(filename)

if missing_files:
    raise FileNotFoundError(
        f"Missing artifact files: {missing_files}"
    )

print("\n[1] Required files: PASSED")

# ------------------------------------------------------------
# 2. Load and validate the FAISS index.
# ------------------------------------------------------------

loaded_index = faiss.read_index(
    os.path.join(
        ARTIFACT_DIR,
        "resume_faiss.index"
    )
)

print(
    f"[2] FAISS index: "
    f"{loaded_index.ntotal} vectors, "
    f"{loaded_index.d} dimensions"
)

if loaded_index.ntotal != len(chunks):
    raise ValueError(
        "FAISS vector count does not match chunk count."
    )

print("[2] FAISS-to-chunks validation: PASSED")

# ------------------------------------------------------------
# 3. Load and validate chunk metadata.
# ------------------------------------------------------------

with open(
    os.path.join(
        ARTIFACT_DIR,
        "chunks.json"
    ),
    "r",
    encoding="utf-8"
) as f:
    loaded_chunks = json.load(f)

if len(loaded_chunks) != loaded_index.ntotal:
    raise ValueError(
        "Loaded chunk count does not match FAISS index."
    )

required_chunk_fields = {
    "candidate_id",
    "chunk_id",
    "retrieval_text"
}

for chunk in loaded_chunks:
    if not required_chunk_fields.issubset(chunk.keys()):
        raise ValueError(
            "A chunk is missing required metadata fields."
        )

print("[3] Chunk metadata validation: PASSED")

# ------------------------------------------------------------
# 4. Load and validate candidate profiles.
# ------------------------------------------------------------

with open(
    os.path.join(
        ARTIFACT_DIR,
        "candidate_profiles.json"
    ),
    "r",
    encoding="utf-8"
) as f:
    loaded_profiles = json.load(f)

if len(loaded_profiles) != 220:
    raise ValueError(
        "Unexpected number of candidate profiles."
    )

for candidate_id, profile in loaded_profiles.items():
    if "candidate_id" not in profile:
        raise ValueError(
            f"Candidate {candidate_id} is missing candidate_id."
        )

    if "resume_text" not in profile:
        raise ValueError(
            f"Candidate {candidate_id} is missing resume_text."
        )

print("[4] Candidate profiles validation: PASSED")

# ------------------------------------------------------------
# 5. Validate chunk candidate IDs against profiles.
# ------------------------------------------------------------

profile_ids = {
    int(candidate_id)
    for candidate_id in loaded_profiles.keys()
}

chunk_candidate_ids = {
    chunk["candidate_id"]
    for chunk in loaded_chunks
}

unknown_candidate_ids = (
    chunk_candidate_ids - profile_ids
)

if unknown_candidate_ids:
    raise ValueError(
        "Chunks reference unknown candidate IDs: "
        f"{sorted(unknown_candidate_ids)}"
    )

print("[5] Candidate ID mapping validation: PASSED")

# ------------------------------------------------------------
# 6. Load and validate configuration.
# ------------------------------------------------------------

with open(
    os.path.join(
        ARTIFACT_DIR,
        "config.json"
    ),
    "r",
    encoding="utf-8"
) as f:
    loaded_config = json.load(f)

if loaded_config["embedding"]["model_name"] != embedding_model_name:
    raise ValueError(
        "Embedding model mismatch."
    )

if loaded_config["embedding"]["dimension"] != loaded_index.d:
    raise ValueError(
        "Embedding dimension mismatch."
    )

if loaded_config["dataset"]["chunk_count"] != len(loaded_chunks):
    raise ValueError(
        "Configuration chunk count mismatch."
    )

if loaded_config["dataset"]["candidate_count"] != len(loaded_profiles):
    raise ValueError(
        "Configuration candidate count mismatch."
    )

print("[6] Configuration validation: PASSED")

# ------------------------------------------------------------
# Final result.
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("ALL PRODUCTION ARTIFACTS VALIDATED SUCCESSFULLY")
print("=" * 60)

print(f"Candidates: {len(loaded_profiles)}")
print(f"Chunks: {len(loaded_chunks)}")
print(f"FAISS vectors: {loaded_index.ntotal}")
print(f"Embedding dimension: {loaded_index.d}")

In [ ]:
from pathlib import Path

# Create the production project structure.
# The notebook remains the development environment,
# while these directories will contain the deployable application.

PROJECT_DIR = Path("/content/rag-talent-search-engine")

project_directories = [
    PROJECT_DIR / "app",
    PROJECT_DIR / "src",
    PROJECT_DIR / "src" / "retrieval",
    PROJECT_DIR / "src" / "ranking",
    PROJECT_DIR / "src" / "evaluation",
    PROJECT_DIR / "src" / "utils",
    PROJECT_DIR / "artifacts",
    PROJECT_DIR / "tests"
]

for directory in project_directories:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

print("=" * 60)
print("PROJECT STRUCTURE CREATED")
print("=" * 60)

for directory in project_directories:
    relative_path = directory.relative_to(PROJECT_DIR)
    print(f"{relative_path}/")

print("\nProject root:")
print(PROJECT_DIR)

In [ ]:
import shutil
from pathlib import Path

# Copy validated production artifacts into the project repository.
# These files will be loaded directly by the Streamlit application.

SOURCE_ARTIFACT_DIR = Path(
    "/content/rag_talent_search_artifacts"
)

TARGET_ARTIFACT_DIR = PROJECT_DIR / "artifacts"

artifact_files = [
    "resume_faiss.index",
    "chunks.json",
    "candidate_profiles.json",
    "config.json"
]

print("=" * 60)
print("COPYING PRODUCTION ARTIFACTS")
print("=" * 60)

for filename in artifact_files:
    source_path = SOURCE_ARTIFACT_DIR / filename
    target_path = TARGET_ARTIFACT_DIR / filename

    if not source_path.exists():
        raise FileNotFoundError(
            f"Source artifact not found: {source_path}"
        )

    shutil.copy2(
        source_path,
        target_path
    )

    print(f"Copied: {filename}")

print("\n" + "=" * 60)
print("ARTIFACT COPY COMPLETED")
print("=" * 60)

for filename in artifact_files:
    target_path = TARGET_ARTIFACT_DIR / filename
    print(
        f"{filename}: "
        f"{target_path.stat().st_size / 1024:.2f} KB"
    )

In [ ]:
from pathlib import Path

# Create the retrieval engine module.
# This module loads the persisted artifacts and provides
# reusable semantic retrieval functionality for the application.

retrieval_engine_path = PROJECT_DIR / "src" / "retrieval" / "engine.py"

retrieval_engine_code = r'''
import json
from collections import defaultdict
from pathlib import Path

import faiss
from sentence_transformers import SentenceTransformer


class RetrievalEngine:
    """
    Semantic retrieval engine for the RAG-powered talent search system.

    The engine loads persisted FAISS vectors, chunk metadata,
    candidate profiles, and embedding configuration.
    """

    def __init__(self, artifact_dir):
        self.artifact_dir = Path(artifact_dir)

        self._load_config()
        self._load_index()
        self._load_chunks()
        self._load_candidate_profiles()
        self._load_embedding_model()

    def _load_config(self):
        config_path = self.artifact_dir / "config.json"

        with open(
            config_path,
            "r",
            encoding="utf-8"
        ) as file:
            self.config = json.load(file)

    def _load_index(self):
        index_path = self.artifact_dir / "resume_faiss.index"

        self.index = faiss.read_index(
            str(index_path)
        )

    def _load_chunks(self):
        chunks_path = self.artifact_dir / "chunks.json"

        with open(
            chunks_path,
            "r",
            encoding="utf-8"
        ) as file:
            self.chunks = json.load(file)

        if len(self.chunks) != self.index.ntotal:
            raise ValueError(
                "Chunk count does not match FAISS index size."
            )

    def _load_candidate_profiles(self):
        profiles_path = (
            self.artifact_dir /
            "candidate_profiles.json"
        )

        with open(
            profiles_path,
            "r",
            encoding="utf-8"
        ) as file:
            self.candidate_profiles = json.load(file)

    def _load_embedding_model(self):
        model_name = self.config["embedding"]["model_name"]

        self.embedding_model = SentenceTransformer(
            model_name
        )

    def semantic_search(
        self,
        query,
        top_k=30
    ):
        """
        Retrieve the most semantically similar resume chunks.
        """

        query_embedding = self.embedding_model.encode(
            [query],
            normalize_embeddings=True,
            convert_to_numpy=True
        ).astype("float32")

        scores, indices = self.index.search(
            query_embedding,
            top_k
        )

        results = []

        for score, index in zip(
            scores[0],
            indices[0]
        ):
            if index == -1:
                continue

            result = self.chunks[index].copy()
            result["similarity_score"] = float(score)

            results.append(result)

        return results

    def retrieve_candidates(
        self,
        query,
        top_k_chunks=30,
        top_k_candidates=5
    ):
        """
        Retrieve and group relevant chunks by candidate.
        """

        chunk_results = self.semantic_search(
            query=query,
            top_k=top_k_chunks
        )

        candidate_groups = defaultdict(list)

        for result in chunk_results:
            candidate_id = result["candidate_id"]

            candidate_groups[
                candidate_id
            ].append(result)

        candidate_results = []

        for candidate_id, candidate_chunks in (
            candidate_groups.items()
        ):
            best_score = max(
                chunk["similarity_score"]
                for chunk in candidate_chunks
            )

            sorted_chunks = sorted(
                candidate_chunks,
                key=lambda item: item["similarity_score"],
                reverse=True
            )

            candidate_results.append({
                "candidate_id": candidate_id,
                "similarity_score": best_score,
                "matched_chunks": sorted_chunks
            })

        candidate_results.sort(
            key=lambda item: item["similarity_score"],
            reverse=True
        )

        return candidate_results[
            :top_k_candidates
        ]

    def get_candidate_profile(
        self,
        candidate_id
    ):
        """
        Return the persisted profile for a candidate.
        """

        profile = self.candidate_profiles.get(
            str(candidate_id)
        )

        if profile is None:
            raise ValueError(
                f"Candidate ID {candidate_id} was not found."
            )

        return profile
'''

retrieval_engine_path.write_text(
    retrieval_engine_code,
    encoding="utf-8"
)

print("=" * 60)
print("RETRIEVAL ENGINE CREATED")
print("=" * 60)

print(f"Path: {retrieval_engine_path}")
print(f"Size: {retrieval_engine_path.stat().st_size} bytes")

In [ ]:
import shutil
from pathlib import Path

# Create a ZIP archive of the current production project.
# This archive contains the project structure, retrieval engine,
# and validated production artifacts.

project_zip_base = Path(
    "/content/rag-talent-search-engine"
)

archive_path = shutil.make_archive(
    str(project_zip_base),
    "zip",
    root_dir=str(PROJECT_DIR)
)

print("=" * 60)
print("PROJECT ARCHIVE CREATED")
print("=" * 60)

print(f"Archive: {archive_path}")

archive_size_mb = (
    Path(archive_path).stat().st_size
    / (1024 ** 2)
)

print(
    f"Archive size: "
    f"{archive_size_mb:.2f} MB"
)

In [ ]:
from google.colab import files

files.download(
    "/content/rag-talent-search-engine.zip"
)